In [1]:
import requests
import json
import pandas as pd


In [2]:
# pull Road Closure feed and create road_events_json object

# un-comment out the following lines when running in VS Code, comment out those below
password = pd.read_csv("passwords.csv")
password_nps = password["password"][0]

# comment out the above when running in DataBricks, un-comment below


road_events_url = "https://developer.nps.gov/api/v1/roadevents?api_key=" + password_nps

response_API = requests.get(road_events_url)
data = response_API.text
road_events_json = json.loads(data)


In [3]:
# pull Alerts feed and create alerts_json object
alerts_url = "https://developer.nps.gov/api/v1/alerts?limit=10000&api_key=" + password_nps

response_API = requests.get(alerts_url)
data = response_API.text
alerts_json = json.loads(data)


In [4]:
# make a dataframe from the data sources section of Road Events
data_sources_df = pd.json_normalize(
    road_events_json, 
    record_path=['road_event_feed_info','data_sources'])

# make a dataframe from the 'properties' section of Road Events
properties_list = [f['properties'] for f in road_events_json['features']]
properties_df = pd.json_normalize(properties_list)


# merge the two dataframes on the 'data_source_id' field
road_closures_feed_df = pd.merge(
    properties_df,                   # The main data (left)
    data_sources_df,                    # The metadata (right)
    left_on='core_details.data_source_id', 
    right_on='data_source_id',
    how='left'                     # Ensures we don't lose any road events
)

# add a column for today's date and make it the first column
today = pd.Timestamp.now().normalize()
road_closures_feed_df.insert(0, 'extraction_date', today)


# save it as a csv with today's date in the file name
today_str = today.strftime('%Y-%m-%d')
road_closures_feed_df.to_csv('Road_Closure_Feeds/' + today_str + '_Road_Closure_Feed.csv', index=False)

# save the raw json too
with open( 'Road_Closure_Feeds/' + today_str + '_road_closures_feed.json', 'w', encoding='utf-8') as f:
    json.dump(road_events_json, f, indent=4)

In [5]:
# make a dataframe out of the Alerts feed
alerts_df = pd.json_normalize(alerts_json, record_path=['data'])

# add a column for today's date and make it the first column
today = pd.Timestamp.now().normalize()
alerts_df.insert(0, 'extraction_date', today)

# save it as a csv with today's date in the file name
alerts_df.to_csv('Alerts_Feeds/' + today_str + '_Alerts_Feed.csv', index=False)

# save the raw json too
with open( 'Alerts_Feeds/' + today_str + '_alerts_feed.json', 'w', encoding='utf-8') as f:
    json.dump(alerts_json, f, indent=4)